<a href="https://colab.research.google.com/github/RAPID-Facility/rAPIdtools/blob/main/examples/asset_analysis_example_generalized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Automated Extraction and AI Analysis of Infrastructure Assets from Multi-Source Imagery Using `rAPIdtools`**

In [ ]:
!pip install "rapidtools[designsafe]"

## <u>Part 1: Automated Extraction of Assets from RAPID Imagery</u>
In this first step, we automatically identify and map all buildings within the disaster zone without dependence on pre-existing GIS databases. By integrating high-resolution aerial imagery with Meta's Segment Anything Model 3 (SAM 3), we construct a foundational asset inventory from the ground up. Although this example emphasizes buildings, the process is applicable to any physical asset discernible in aerial imagery.

### Define the assets you would like to detect & Define Gemini API Key

In [ ]:
import os

TARGET_ASSET = 'tree' # free-form text
APPRX_ASSET_SIZE=50 # in meters
DETECT_IN_RECON_IMAGERY= False # Set to True if the asset can only be detected in post-disaster imagery

# DesignSafe publication PRJ-5815 (DOI 10.17603/ds2-wsvx-7s97): pick a tile and a patch.
TILE = 'Eaton_Trinity38_RGBortho_20250214.tiff'  # 4.7 GB; the other tiles are 10 to 20 GB
PATCH_SIZE_M = 200  # side of the square patch clipped from the tile, in meters
PATCH_RESOLUTION_M = 0.05  # resample the patch to 5 cm pixels; the tiles are 1.9 cm
PATCH_CENTER = None  # (lon, lat); None uses the tile center
STEM = TILE.rsplit('.', 1)[0]

API_KEY = os.environ.get('GOOGLE_API_KEY', 'PASTE_YOUR_GEMINI_API_KEY')  # never commit a real key

### Import required rAPIdtools methods and packages

In [ ]:
from rapidtools import (
    AerialImageryExtractor,
    BingOrthomosaicExtractor,
    BoundingBox,
    BuildingRegularizer,
    GeminiAssetAnalyzer,
    MapillaryImageExtractor,
    SAM3OrthoFeatureExtractor,
    Pipeline,
    PolygonRegion,
    PhysicalAssetCollection,
    RoadwayRegularizer,
    download_dataset
)
from rapidtools.data_sources.designsafe import (
    EATON_FIRE_IMAGERY_URI,
    list_published_files,
    patch_from_published,
)

from pathlib import Path

### Clip a patch from a UW RAPID post-event orthomosaic published on DesignSafe (Eaton Fire, February 2025) and determine its bounds

The tiles in [PRJ-5815](https://doi.org/10.17603/ds2-wsvx-7s97) are 4.7 to 20 GB each. `patch_from_published` reads the tile in place on DesignSafe JupyterHub (`~/NHERI-Published`) and writes only the requested window, resampled from the tile's overviews to `PATCH_RESOLUTION_M`, so nothing large is copied or decoded. Off DesignSafe it downloads with dapi, but refuses files above 2 GB; raise `max_download_gb` if you accept the transfer.

In [ ]:
tiles = list_published_files(EATON_FIRE_IMAGERY_URI)
print('\n'.join(f"{t['name']:45s} {t['size_gb']:5.1f} GB" for t in tiles))

raster_path = patch_from_published(
    f'{EATON_FIRE_IMAGERY_URI}/{TILE}',
    f'data/{STEM}_patch.tiff',
    center=PATCH_CENTER,
    size_m=PATCH_SIZE_M,
    resolution_m=PATCH_RESOLUTION_M,
)
region_of_interest = BoundingBox.from_raster(raster_path)
region_of_interest

### Extract preliminary asset geometries using Meta's Segment Anything Model (SAM 3)

In [ ]:
sam_extractor = SAM3OrthoFeatureExtractor(
    prompt=TARGET_ASSET,
    patch_size=APPRX_ASSET_SIZE,
    unit='meters',
    overlap_ratio=0.25,
    batch_size=4,
    threshold=0.50,
    mask_threshold=0.40,
)

if DETECT_IN_RECON_IMAGERY:
    prelim_collection = sam_extractor(raster_path)
else:
  bing_extractor = BingOrthomosaicExtractor(max_workers=10)
  bing_raster_path = bing_extractor(
      region=region_of_interest,
      output_path=f'{STEM}_bing.tiff'
  )
  prelim_collection = sam_extractor(bing_raster_path)

_ = prelim_collection.to_geojson(f'{STEM}_assets_preliminary.geojson')

### Extract buffered image crops to provide spatial context for geometric regularization


In [ ]:
if DETECT_IN_RECON_IMAGERY:
  extractor = AerialImageryExtractor(
      dataset=raster_path,
      save_directory='output/asset_crops',
      buffer_asset='20 m',
      force_square_image=True,
  )
else:
  extractor = AerialImageryExtractor(
      dataset=bing_raster_path,
      save_directory='output/asset_crops',
      buffer_asset='20 m',
      force_square_image=True,
  )
assets_with_images = extractor(prelim_collection)

### Refine and regularize asset geometries into clean, GIS-ready polygons


In [ ]:
if 'building' in TARGET_ASSET:
    regularizer = BuildingRegularizer(batch_size=8)
elif 'road' in TARGET_ASSET:
    regularizer = RoadwayRegularizer()
else:
  regularizer = None

if regularizer:
  final_assets = regularizer(assets_with_images)
else:
  final_assets = assets_with_images

_ = final_assets.to_geojson(f'{STEM}_assets_final.geojson')

## <u>Part 2: Aerial Post-Disaster Condition Assessment</u>

### Configure the extraction and analysis pipeline

In [ ]:
[prompt_path] = download_dataset('aerial_chs_prompts')
damage_classification_prompt = Path(prompt_path).read_text().strip()

pipeline = Pipeline()

extractor = AerialImageryExtractor(
    dataset=raster_path,
    save_directory='eaton_fire_aerial_feb25/overlaid_imagery',
    overlay_asset_outline=True,
    image_prefix=STEM,
    keep_multiple_copies=True,
)

analyzer = GeminiAssetAnalyzer(
    api_key=API_KEY,
    prompt=damage_classification_prompt,
)

### Execute the workflow and export results

In [ ]:
pipeline.add_step(extractor)
pipeline.add_step(analyzer)

processed_collection = pipeline.run(final_assets)

final_collection = processed_collection.filter_empty()

_ = final_collection.to_geojson(
    f'{STEM}_assets_inferred.geojson',
    ignore_properties=['image_assets']
)

### Save the results to My Data (optional)

Set `UPLOAD_TO` to a folder in your DesignSafe My Data to copy the GeoJSON outputs there with dapi, for example `tapis://designsafe.storage.default/<username>/eaton-assets`.

In [ ]:
UPLOAD_TO = None

if UPLOAD_TO:
    from dapi import DSClient

    ds = DSClient()
    for name in [f'{STEM}_assets_preliminary.geojson', f'{STEM}_assets_final.geojson', f'{STEM}_assets_inferred.geojson']:
        if Path(name).exists():
            ds.files.upload(name, f'{UPLOAD_TO.rstrip("/")}/{name}')
    print(f'Uploaded results to {UPLOAD_TO}')